# Baseline — Vietnamese Word Segmentation

Notebook chạy từ thư mục gốc repository. Cần có `data/processed/train.csv`,
`dev.csv`, `test.csv` và các module trong `src/`.

**Mục tiêu:** kiểm tra định dạng dữ liệu, chạy bốn baseline,
sinh dự đoán trên test và lưu CSV để dùng chung pipeline đánh giá.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path("..").resolve()))

import pandas as pd
import torch

print("PyTorch:", torch.__version__, "| device:", "cuda" if torch.cuda.is_available() else "cpu")

TRAIN = Path("../data/processed/train.csv")
DEV   = Path("../data/processed/dev.csv")
TEST  = Path("../data/processed/test.csv")

Working directory: d:\UEH Academic Life\NLP\Seminar\GIT\vietnamese-word-segmentation\notebooks
PyTorch: 2.12.0+cpu | device: cpu


## 1. Đọc và kiểm tra dữ liệu

In [20]:
required = {"id", "text", "segmented", "labels"}
train_df = pd.read_csv(TRAIN).fillna("")
dev_df   = pd.read_csv(DEV).fillna("")
test_df  = pd.read_csv(TEST).fillna("")

for name, df in [("train", train_df), ("dev", dev_df), ("test", test_df)]:
    assert required.issubset(df.columns), f"{name}: missing columns"
    lengths_ok = df.apply(
        lambda r: len(str(r["text"]).split()) == len(str(r["labels"]).split()), axis=1
    )
    tags_ok = df["labels"].apply(
        lambda s: set(str(s).split()).issubset({"B", "I"})
    )
    print(name, "rows=", len(df), "length alignment=", lengths_ok.all(),
          "valid tags=", tags_ok.all())
    assert lengths_ok.all() and tags_ok.all()

train_df.head()

train rows= 1400 length alignment= True valid tags= True
dev rows= 1123 length alignment= True valid tags= True
test rows= 800 length alignment= True valid tags= True


,id,text,segmented,labels
0,1,Tôi nhớ lời anh chủ tịch xã Bùi Văn Luyến nhắc...,Tôi nhớ lời anh chủ_tịch xã Bùi_Văn_Luyến nhắc...,B B B B B I B B I I B B B B B B B I B B B I B B
1,2,"Hiện nay xã có 68 tổ nhân dân , mỗi tổ phụ trá...","Hiện_nay xã có 68 tổ nhân_dân , mỗi tổ phụ_trá...",B I B B B B B I B B B B I B B I B
2,3,"Chủ trương tốt nhưng dân không hiểu , không hư...","Chủ_trương tốt nhưng dân không hiểu , không hư...",B I B B B B B B B B I B B B I B
3,4,Nên trước nhất người đảng viên phải làm gương .,Nên trước_nhất người đảng_viên phải làm_gương .,B B I B B I B B I B
4,5,"Riêng về xuất khẩu lao động , anh Luyến có các...","Riêng về xuất_khẩu lao_động , anh Luyến có các...",B B B I B I B B B B B B B B B B I I B B I B B ...


## 2. Baseline: fit từ điển/ngram trên chỉ tập train

In [21]:
from src.baseline import (
    WhitespaceSegmenter, RegexSegmenter, MaximumMatchingSegmenter,
    BigramSegmenter, build_word_dictionary, build_bigram_model
)

dictionary = build_word_dictionary(train_df["segmented"])
bigram_lm = build_bigram_model(train_df["segmented"], alpha=0.1)

baseline_models = {
    "whitespace": WhitespaceSegmenter(),
    "regex": RegexSegmenter(),
    "maximum_matching": MaximumMatchingSegmenter(dictionary),
    "bigram": BigramSegmenter(dictionary, bigram_lm),
}

sample = "Hiện nay xã có 68 tổ nhân dân ."
for name, model in baseline_models.items():
    pred_seg, pred_tags = model.predict_sentence(sample)
    print(f"{name:20s}: {pred_seg} | {' '.join(pred_tags)}")

whitespace          : Hiện nay xã có 68 tổ nhân dân . | B B B B B B B B B
regex               : Hiện nay xã có 68 tổ nhân dân . | B B B B B B B B B
maximum_matching    : Hiện_nay xã có 68 tổ nhân_dân . | B I B B B B B I B
bigram              : Hiện_nay xã có 68 tổ nhân_dân . | B I B B B B B I B


## 3. Sinh dự đoán baseline cho test

In [18]:
baseline_outputs = test_df[["id", "text", "segmented", "labels"]].copy()
for name, model in baseline_models.items():
    predictions = [model.predict_sentence(t) for t in test_df["text"]]
    baseline_outputs[f"{name}_segmented"] = [x[0] for x in predictions]
    baseline_outputs[f"{name}_labels"] = [" ".join(x[1]) for x in predictions]

Path("outputs/baseline").mkdir(parents=True, exist_ok=True)
baseline_outputs.to_csv("outputs/baseline/test_predictions.csv",
                        index=False, encoding="utf-8-sig")
baseline_outputs.head()

,id,text,segmented,labels,whitespace_segmented,whitespace_labels,regex_segmented,regex_labels,maximum_matching_segmented,maximum_matching_labels,bigram_segmented,bigram_labels
0,1,"Thanh bắt chuyện với Hùng và nói : "" Tôi trông...","Thanh bắt_chuyện với Hùng và nói : "" Tôi trông...",B B I B B B B B B B B B B I B B B,"Thanh bắt chuyện với Hùng và nói : "" Tôi trông...",B B B B B B B B B B B B B B B B B,"Thanh bắt chuyện với Hùng và nói : "" Tôi trông...",B B B B B B B B B B B B B B B B B,"Thanh bắt_chuyện với Hùng và nói : "" Tôi trông...",B B I B B B B B B B B B B B B B B,"Thanh bắt_chuyện với Hùng và nói : "" Tôi trông...",B B I B B B B B B B B B B B B B B
1,2,"Hùng giật mình : "" Sao tôi không biết ông nhỉ ...","Hùng giật_mình : "" Sao tôi không biết ông nhỉ ...",B B I B B B B B B B B B B B,"Hùng giật mình : "" Sao tôi không biết ông nhỉ ...",B B B B B B B B B B B B B B,"Hùng giật mình : "" Sao tôi không biết ông nhỉ ...",B B B B B B B B B B B B B B,"Hùng giật_mình : "" Sao tôi không biết ông nhỉ ...",B B I B B B B B B B B B B B,"Hùng giật_mình : "" Sao tôi không biết ông nhỉ ...",B B I B B B B B B B B B B B
2,3,"Thanh quay sang Hùng : "" ông quê đâu nhỉ ?","Thanh quay sang Hùng : "" ông quê đâu nhỉ ?",B B B B B B B B B B B,"Thanh quay sang Hùng : "" ông quê đâu nhỉ ?",B B B B B B B B B B B,"Thanh quay sang Hùng : "" ông quê đâu nhỉ ?",B B B B B B B B B B B,"Thanh quay sang Hùng : "" ông quê đâu nhỉ ?",B B B B B B B B B B B,"Thanh quay sang Hùng : "" ông quê đâu nhỉ ?",B B B B B B B B B B B
3,4,"À , tôi còn có danh sách lớp đây này "" .","À , tôi còn có danh_sách lớp đây này "" .",B B B B B B I B B B B B,"À , tôi còn có danh sách lớp đây này "" .",B B B B B B B B B B B B,"À , tôi còn có danh sách lớp đây này "" .",B B B B B B B B B B B B,"À , tôi còn có danh sách lớp đây này "" .",B B B B B B B B B B B B,"À , tôi còn có danh sách lớp đây này "" .",B B B B B B B B B B B B
4,5,Nói rồi Thanh giở danh sách lớp đọc vanh vách ...,Nói rồi Thanh giở danh_sách lớp đọc vanh_vách ...,B B B B B I B B B I B I B B I B B I B B B,Nói rồi Thanh giở danh sách lớp đọc vanh vách ...,B B B B B B B B B B B B B B B B B B B B B,Nói rồi Thanh giở danh sách lớp đọc vanh vách ...,B B B B B B B B B B B B B B B B B B B B B,Nói rồi Thanh giở danh sách lớp đọc vanh vách ...,B B B B B B B B B B B I B B I B B I B B B,Nói rồi Thanh giở danh sách lớp đọc vanh vách ...,B B B B B B B B B B B I B B I B B I B B B
